프로젝트 수행자 : 이다겸  
프로젝트 수행일 : 2026.08.24

---

모델 배포 개론 08  
Last modified : 2026.03   
작성 : 박광성 (모두의연구소)  
수정 : 김지성 박기웅 (모두의연구소)  

# Day 8 — 자율 프로젝트: 나만의 모델 서빙 서비스 만들기

---

> **오늘의 목표**
>
> Day 1~7에서 배운 기술을 조합하여, 본인이 관심 있는 도메인의 모델을 서빙하는 서비스를 직접 만듭니다.  

---



## 1. 프로젝트 요구사항

---

### 1.1 조건

Day 5(주택 가격 예측)와 Day 6~7(이미지 분류 / 챗봇)에서 만든 서비스와 **동일한 구조**를 기본 베이스로 합니다.

```
필수 구현 항목:

1. FastAPI 백엔드
   - 추론 엔드포인트 (POST /predict)
   - Pydantic으로 입력 검증
   - 비동기 추론 (run_in_executor)

2. API Key 인증
   - Day 6의 auth.py 재사용

3. Streamlit 프론트엔드
   - 사용자 입력 → API 호출 → 결과 표시

4. 에러 처리
   - 잘못된 입력, 모델 에러 시 적절한 HTTP 상태 코드 반환
```



### 1.2 제한 사항

```
하지 않는 것:

- 모델 학습 (사전학습 모델을 가져다 씁니다)
- Docker 패키징 (MLOps 과정에서 다룹니다)
- 데이터베이스 연동
```



### 1.3 평가 기준

```
✅ 서버가 정상적으로 실행되는가?
✅ Swagger UI에서 추론이 동작하는가?
✅ API Key 없이 요청하면 401이 반환되는가?
✅ 잘못된 입력에 대해 적절한 에러 메시지가 나오는가?
✅ Streamlit UI에서 입력 → 결과 확인이 가능한가?
```

In [2]:
# 서버 실행 도우미 — 노트북 맨 처음에 한 번 실행하세요.
# 노트북 안에서 uvicorn 서버를 띄우고 멈추는 함수를 정의합니다.
import os, sys, asyncio, threading, time, socket, contextlib
import uvicorn

# 작업 디렉터리를 app/ 가 있는 위치로 맞춥니다 (notebooks/ 안에서 열어도 동작).
if not os.path.isdir('app') and os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
# 코드를 저장할 폴더를 미리 만들어 둡니다.
for _d in ('app', 'models', 'data', 'frontend'):
    os.makedirs(_d, exist_ok=True)

_SERVERS = {}  # port -> (server, thread)

def _port_open(host, port):
    with contextlib.closing(socket.socket()) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0

def stop_server(port=8000):
    """실행 중인 서버를 멈춥니다."""
    entry = _SERVERS.pop(port, None)
    if not entry:
        return
    server, thread = entry
    server.should_exit = True
    for _ in range(50):
        if not thread.is_alive():
            break
        time.sleep(0.1)

def serve_in_thread(app, host='127.0.0.1', port=8000, log_level='warning'):
    """백그라운드에서 uvicorn 서버를 띄웁니다.

    app: FastAPI 객체 또는 'app.main:app' 같은 import 경로.
    같은 포트에 서버가 이미 있으면 먼저 멈추고 새로 띄웁니다.
    """
    stop_server(port)
    if _port_open(host, port):
        print(f'⚠️ 포트 {port}를 다른 프로세스가 사용 중입니다 (다른 노트북의 서버일 가능성).')
        print('   그 노트북에서 stop_server(8000)을 실행하거나 커널을 종료한 뒤, 이 셀을 다시 실행하세요.')
        return None
    if isinstance(app, str):
        sys.modules.pop(app.split(':')[0], None)   # 파일을 다시 저장한 경우 최신 내용 반영
    for _ in range(50):
        if not _port_open(host, port):
            break
        time.sleep(0.1)
    config = uvicorn.Config(app, host=host, port=port, log_level=log_level, loop='asyncio')
    server = uvicorn.Server(config)
    server.install_signal_handlers = lambda: None
    def _run():
        # Windows는 SelectorEventLoop, 그 외는 기본 이벤트 루프를 사용합니다.
        if sys.platform == 'win32':
            loop = asyncio.SelectorEventLoop()
        else:
            loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        loop.run_until_complete(server.serve())
    thread = threading.Thread(target=_run, daemon=True)
    thread.start()
    _SERVERS[port] = (server, thread)
    # 모델 로드 때문에 기동이 느릴 수 있다 — 최대 5분 대기 (첫 실행은 다운로드 포함)
    for i in range(600):
        if _port_open(host, port):
            print(f'서버 실행됨: http://{host}:{port}')
            return server
        if not thread.is_alive():
            print('서버 스레드가 종료됐습니다. 위 로그를 확인하세요.')
            return server
        if i > 0 and i % 20 == 0:
            print(f'  ... 모델 로드 중 ({i//2}초 경과)')
        time.sleep(0.5)
    print('5분 내에 서버가 시작되지 않았습니다. 위 로그를 확인하세요.')
    return server

print('서버 도우미 준비 완료 (serve_in_thread, stop_server)')

서버 도우미 준비 완료 (serve_in_thread, stop_server)


---

## 2. 모델 선택 가이드

---

### 2.1 Hugging Face에서 모델 찾기

이번 프로젝트에서 선택한 모델은 텍스트의 감정을 분석할 수 있는 사전 학습 모델인 [snunlp/KR-FinBert-SC](https://huggingface.co/snunlp/KR-FinBert-SC)입니다.  
모델의 크기는 406 MB 으로 로컬에서 수행하기 적합한 모델입니다. 또한 Hugging Face의 pipeline()을 사용해 볼 수 있는 모델이라 본 프로젝트에 적합합니다. 


### 2.1 모델 동작 확인

**서버 코드를 작성하기 전에** 노트북에서 먼저 동작을 확인합니다.

In [ ]:
# transformers 가 없으면 설치합니다. (Colab은 세션마다 확인이 필요합니다)
import importlib.util, sys, subprocess

for _pkg in ("transformers", "accelerate"):
    if importlib.util.find_spec(_pkg) is None:
        print(f"❌ {_pkg} 미설치 → 지금 설치합니다.")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=True)

# 예시: 텍스트 감정 분석
from transformers import pipeline

# 본인이 선택한 모델로 교체하세요
classifier = pipeline("text-classification", model="snunlp/KR-FinBert-SC")

result = classifier("오늘 주가가 크게 올랐습니다")
print(result)
# [{'label': 'positive', 'score': 0.9987346529960632}]


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': 'positive', 'score': 0.9987346529960632}]



---

## 3. 프로젝트 뼈대 코드

---



### 3.1 폴더 구조

In [19]:
import os

dirs = ["app", "models", "frontend"]
for d in dirs:
    os.makedirs(d, exist_ok=True)

print("프로젝트 구조:")
print("""
my-project/
├── 📁 app/
│   ├── auth.py              ← Day 6에서 만든 것 그대로 재사용
│   ├── schemas_pj2.py           ← 입력/출력 스키마 정의 (직접 작성)
│   ├── model_service_pj2.py     ← 모델 로드 + 추론 함수 (직접 작성)
│   └── main_pj2.py              ← FastAPI 서버 (직접 작성)
│
├── 📁 frontend/
│   └── app_pj2.py               ← Streamlit UI (직접 작성)
│
└── requirements.txt
""")

프로젝트 구조:

my-project/
├── 📁 app/
│   ├── auth.py              ← Day 6에서 만든 것 그대로 재사용
│   ├── schemas_pj2.py           ← 입력/출력 스키마 정의 (직접 작성)
│   ├── model_service_pj2.py     ← 모델 로드 + 추론 함수 (직접 작성)
│   └── main_pj2.py              ← FastAPI 서버 (직접 작성)
│
├── 📁 frontend/
│   └── app_pj2.py               ← Streamlit UI (직접 작성)
│
└── requirements.txt



### 3.2 auth.py — 재사용

In [20]:
%%writefile app/auth.py
"""
Day 6에서 만든 인증 모듈을 그대로 재사용합니다.
"""
from fastapi import HTTPException, Header

VALID_API_KEYS = {
    "test-key-001": "사용자A",
    "test-key-002": "사용자B",
}


async def verify_api_key(x_api_key: str = Header(None)) -> str:
    if x_api_key is None:
        raise HTTPException(
            status_code=401,
            detail="API Key가 필요합니다. X-API-Key 헤더를 포함해 주세요.",
        )
    if x_api_key not in VALID_API_KEYS:
        raise HTTPException(
            status_code=401,
            detail="유효하지 않은 API Key입니다.",
        )
    return VALID_API_KEYS[x_api_key]

Overwriting app/auth.py


### 3.3 schemas_pj2.py — 직접 작성

In [27]:
%%writefile app/schemas_pj2.py
"""
입력/출력 스키마를 정의하세요.

참고: Day 2 섹션 4 (Pydantic 기초), Day 5 섹션 3 (HousingRequest)

TODO:
  - 본인의 모델 입력에 맞는 Request 스키마를 정의하세요.
  - 모델 출력에 맞는 Response 스키마를 정의하세요.
  - 필수 필드와 선택 필드를 구분하세요.
  - 적절한 검증 규칙(타입, 범위, 길이 등)을 추가하세요.

예시 (텍스트 분류의 경우):
  class PredictRequest(BaseModel):
      text: str = Field(..., min_length=1, max_length=5000)

  class PredictResponse(BaseModel):
      success: bool
      label: str
      confidence: float
"""
from pydantic import BaseModel, Field

# ── 여기에 작성하세요 ──────────────────────────
class PredictRequest(BaseModel):
  text: str = Field(..., min_length=1, max_length=5000, description="분석할 텍스트를 입력하세요")
  @field_validator('text')
  def validate_text(cls, v):
    if not v.strip():
        raise ValueError("텍스트는 공백만으로 구성될 수 없습니다.")
    return v

class PredictResponse(BaseModel):
  success: bool = Field(..., description="예측 성공 여부")
  label: str = Field(..., description="예측된 레이블")
  confidence: float = Field(..., ge=0.0, le=1.0, description="예측 신뢰도 (0~1 사이)")


Writing app/schemas_pj2.py


### 3.4 model_service_pj2.py — 직접 작성

In [ ]:
%%writefile app/model_service_pj2.py
"""
모델 로드와 추론 함수를 정의하세요.

참고: Day 1 섹션 5 (모델 로드), Day 5 섹션 2 (HousingPredictor)

TODO:
  1. load_model(): 모델을 로드하여 반환합니다.
     - transformers의 pipeline() 또는 from_pretrained()을 사용합니다.
     - 섹션 2.3에서 확인한 코드를 여기에 옮기면 됩니다.

  2. predict(model, input_data): 입력을 받아 추론 결과를 반환합니다.
     - 입력 전처리가 필요하면 여기서 합니다.
     - 결과를 dict로 반환합니다.

예시 (텍스트 분류의 경우):
  def load_model():
      return pipeline("text-classification", model="모델이름")

  def predict(model, text: str) -> dict:
      result = model(text)
      return {"label": result[0]["label"], "confidence": result[0]["score"]}
"""

# ── 여기에 작성하세요 ──────────────────────────
def load_model():
  return pipeline("text-classification", model="snunlp/KR-FinBert-SC")

def predict(model, text: str) -> dict:
  result = model(text)
  return {"label": result[0]["label"], "confidence": result[0]["score"]}

Overwriting app/model_service_pj2.py


### 3.5 main_pj2.py — 직접 작성

In [ ]:
%%writefile app/main_pj2.py
"""
FastAPI 서버를 정의하세요.

참고: Day 5 섹션 3 (housing_api.py), Day 6 섹션 6 (image_api.py)

TODO:
  1. FastAPI 앱 생성
  2. startup 이벤트에서 모델 로드
  3. GET /health 엔드포인트
  4. POST /predict 엔드포인트
     - Pydantic 스키마로 입력 검증
     - Depends(verify_api_key)로 인증 적용
     - run_in_executor로 비동기 추론

필수 import:
  import asyncio
  from concurrent.futures import ThreadPoolExecutor
  from fastapi import FastAPI, Depends, HTTPException
  from app.auth import verify_api_key
  from app.schemas import PredictRequest, PredictResponse  (본인이 정의한 이름)
  from app.model_service import load_model, predict
"""

# ── 여기에 작성하세요 ──────────────────────────
import asyncio
from concurrent.futures import ThreadPoolExecutor
from fastapi import FastAPI, Depends, HTTPException
from contextlib import asynccontextmanager # 추가된 부분

from app.auth import verify_api_key
from app.schemas_pj2 import PredictRequest, PredictResponse
from app.model_service_pj2 import load_model, predict
from app.logger_config import setup_logger
from app.error_handlers import register_error_handlers
from app.middleware import RequestLoggingMiddleware

# ==== 설정 ====
logger = setup_logger("model_serving_project_2")

# 추론 전용 스레드풀 
inference_executor = ThreadPoolExecutor(max_workers=2, thread_name_prefix="inference")

# 전역 모델 변수
inference_model = None 

# 서버 생애주기 (lifespan)
@asynccontextmanager
async def lifespan(app: FastAPI):
    global inference_model
    import torch
    
    model_name = "snunlp/KR-FinBert-SC"
    logger.info(f"텍스트 분류 모델 로드 중: {model_name}")
    
    # model_service_pj2.py 에 정의한 load_model() 함수를 사용합니다.
    inference_model = load_model() 
    
    logger.info("모델 로드 완료")
    yield


# FastAPI 앱 생성
app = FastAPI(
    title="Text Emotion API",
    description="텍스트의 감정을 분석하는 API",
    version="1.0.0",
    lifespan=lifespan,
)

app.add_middleware(RequestLoggingMiddleware)
register_error_handlers(app)

# ==== 텍스트 분류를 위한 스레드 함수 ====
def run_classification(text: str):
    """별도 스레드에서 실행되는 추론 함수 (LLM 파라미터 제거)"""
    if inference_model is None:
        raise RuntimeError("모델이 로드되지 않았습니다")
        
    # model_service_pj2.py 에 정의한 predict() 함수를 사용합니다.
    return predict(inference_model, text)


# ===== 엔드포인트 =====

@app.get("/health", tags=["System"])
async def health_check():
  return {
          "status": "healthy" if inference_model else "loading"
      }

@app.post("/predict", response_model=PredictResponse, tags=["Prediction"])
async def predict_endpoint( 
    request: PredictRequest,
    user: str = Depends(verify_api_key),
):
    """텍스트 감정 분석 엔드포인트"""
    try:
        loop = asyncio.get_event_loop()
        
        # request.text 만을 전달하여 분석합니다.
        result = await loop.run_in_executor(
            inference_executor, run_classification, request.text
        )
    except Exception as e:
        logger.error(f"추론 중 오류 발생: {e}")
        raise HTTPException(status_code=500, detail="서버 내부 오류가 발생했습니다.")
        
    # predict() 함수의 리턴값 구조에 맞게 수정하세요. (예: result["label"], result["confidence"])
    return PredictResponse(
        success=True, 
        label=result["label"], 
        confidence=result["confidence"]
    )

Writing app/main_pj2.py


### 3.6 frontend/app_pj2.py — 직접 작성

In [6]:
%%writefile frontend/app_pj2.py
"""
Streamlit 프론트엔드를 정의하세요.

참고: Day 4 섹션 6 (대시보드), Day 5 섹션 4 (주택 가격 UI)

TODO:
  1. st.title()로 제목
  2. 사이드바에 API Key 입력
  3. 본인의 모델에 맞는 입력 위젯 (text_input, file_uploader 등)
  4. 버튼 클릭 시 requests.post()로 API 호출
  5. 결과 표시

실행 방법:
  streamlit run frontend/app.py
"""
import streamlit as st
import requests

# ── 여기에 작성하세요 ──────────────────────────

# ===== 페이지 설정 =====
st.set_page_config(
    page_title="텍스트 감정 분석",
    page_icon="📝", # 깨진 이모지 수정
    layout="wide",
)

# ===== API 호출 함수 =====
API_BASE = "http://localhost:8000"
def call_api(text: str, api_key: str):
    """감정 분석 결과를 받아오는 함수"""
    try:
        resp = requests.post(
            f"{API_BASE}/predict",
            json={"text": text},
            headers={"X-API-Key": api_key},            # 인증 헤더
        )
        resp.raise_for_status()
        return resp.json()
    except requests.exceptions.ConnectionError:
        st.error("🔌 **서버에 연결할 수 없습니다.**")
        return None
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 401:
            st.error("🔑 **인증 실패.** API Key를 확인하세요.")
        elif e.response.status_code == 422:
            st.error("🚫 **입력 오류.** 텍스트 길이가 5000자를 넘었거나 비어있습니다.")
        else:
            st.error(f"❌ **서버 에러** (HTTP {e.response.status_code})")
        return None
    except Exception as e:
        st.error(f"❌ **오류:** {type(e).__name__}")
        return None

# ===== 사이드바 =====
with st.sidebar:
    st.header("🔑 인증 설정")
    api_key = st.text_input("API Key", value="test-key-001", type="password")
    st.divider()
    # 서버 상태
    try:
        health = requests.get(f"{API_BASE}/health", timeout=3).json()
        if health.get("status") == "healthy":
            st.success(f"🟢 서버 연결됨")
            st.caption(f"모델: {health.get('model', 'N/A')}")
        else:
            st.warning("🟡 모델 로딩 중...")
    except Exception:
        st.error("🔴 서버 연결 실패")
    st.divider()
    st.caption("Korean Text Emotion Analysis v1.0")
    
# ===== 메인 영역 =====
st.title("📝 텍스트 감정 분석기")
st.markdown("입력하신 텍스트가 긍정적인지 부정적인지 분석해 드립니다.")
# 텍스트 입력창 
user_text = st.text_area("분석할 텍스트를 입력하세요:", height=150)
# 분석 버튼 
if st.button("감정 분석 시작", type="primary"):
  # 예외 처리
  ## API Key 미입력
  if not api_key:
      st.warning("🔑 API Key를 입력하세요.")
  ## 텍스트 미입력
  elif not user_text.strip():
      st.warning("📝 분석할 텍스트를 입력하세요.")
  else:
      with st.spinner("분석 중..."):
          # API 호출 함수 실행 
          result = call_api(user_text, api_key)
          # 결과가 성공적으로 돌아왔을 경우 화면 표시 
          if result and result.get("success"):
              label = result.get("label")
              confidence = result.get("confidence")
              st.success("✅ 분석을 완료하였습니다. ")
              # 결과 강조 
              st.metric(label = "분석 결과", value = label)
              st.progress(confidence, text=f"확률: {confidence:.1%}")

Overwriting frontend/app_pj2.py


---

## 4. 작업 시간

---

### 4.1 권장 순서



```
Step 1. 모델 선택 + 노트북에서 동작 확인 (섹션 2.3)
        → "이 모델이 내 입력에 대해 결과를 반환하는가?"

Step 2. schemas.py 작성
        → "입력과 출력의 형태를 정의"

Step 3. model_service.py 작성
        → "모델 로드 + 추론 함수"

Step 4. main.py 작성
        → "FastAPI 서버 조립"

Step 5. 서버 실행 + Swagger UI 테스트
        → "API가 동작하는가?"

Step 6. frontend/app.py 작성
        → "Streamlit UI 연결"
```



### 4.2 서버 실행 (Step 5에서 사용)

> ⚠️ **코드를 수정했는데 반영이 안 될 때 — 커널을 재시작하세요.**
>
> `app/main.py`, `app/model_service.py` 등을 고친 뒤 아래 셀을 다시 실행해도,
> 이미 메모리에 올라간 이전 코드가 남아 변경이 반영되지 않을 수 있습니다.
> 코드를 수정했다면 **커널 재시작(Kernel → Restart) 후 맨 위 셀부터 다시 실행**하세요.

In [3]:
# 서버 실행 (같은 포트에 서버가 떠 있으면 자동으로 멈추고 새로 띄웁니다)
# app/main.py 를 먼저 불러와 보고, 문제가 있으면 원인을 그대로 보여줍니다.
import importlib, sys, traceback

sys.modules.pop("app.main_pj2", None)          # 파일을 고친 경우 최신 내용 반영
ready = False
try:
    _main = importlib.import_module("app.main_pj2")
    if hasattr(_main, "app"):
        ready = True
    else:
        print("❌ app/main_pj2.py 에 FastAPI 객체 `app` 이 없습니다 — 3.5를 먼저 완성하세요.")
except Exception:
    print("❌ app/main_pj2.py 를 불러올 수 없습니다 — 아래 오류를 확인하세요:")
    traceback.print_exc()

if ready:
    serve_in_thread("app.main_pj2:app", port=8000)


2026-08-24 12:39:39 INFO     [model_serving_project_2] 텍스트 분류 모델 로드 중: snunlp/KR-FinBert-SC


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-08-24 12:39:42 INFO     [model_serving_project_2] 모델 로드 완료
서버 실행됨: http://127.0.0.1:8000


#### Swagger UI 열기

서버가 떴으면 Swagger UI에서 API를 직접 호출해 볼 수 있습니다.  

- 로컬: 브라우저에서 http://localhost:8000/docs

**호출 화면 캡쳐** 

<img src="./img/1.png">



### 4.3 API 테스트 템플릿 (Step 5에서 사용)

In [ ]:
import requests

API_URL = "http://localhost:8000"
HEADERS = {"X-API-Key": "test-key-001"}

# health check
print(requests.get(f"{API_URL}/health").json())

# 추론 테스트 — 본인의 입력에 맞게 수정하세요
response = requests.post(
    f"{API_URL}/predict",
    json={"text": "감독 각본 배우 미술 음악 사운드 모든 요소요소가 장인정신의 결정체. 이게 영화다."},   # 네이버 오디세이 감상평 
    headers=HEADERS,
)
print(f"상태: {response.status_code}")
print(f"결과: {response.json()}")

{'status': 'healthy'}
상태: 200
결과: {'success': True, 'label': 'neutral', 'confidence': 0.9998099207878113}


### 4.4 프론트엔드 실행 (Step 6에서 사용)

`frontend/app_pj2.py`를 작성한 뒤 아래 셀로 띄웁니다.  
노트북 셀에서 `streamlit run`을 직접 실행하면 셀이 끝나지 않으니, 백그라운드로 실행합니다.


In [9]:
# Colab은 세션이 새로 뜰 때마다 streamlit 설치가 필요합니다.
try:
    import streamlit
except ImportError:
    print("❌ Streamlit 미설치 → 지금 설치합니다. (1분 정도 걸립니다)")
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "streamlit"], check=True)

import sys, subprocess, time, socket, contextlib, tempfile, os

try:
    from google.colab import output as _colab_output   # Colab이면 import에 성공합니다
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def run_streamlit(script, port=8501):
    """Streamlit을 백그라운드로 띄우고 '실제로 떴는지'까지 확인한다. (Windows/macOS/Linux 공통)"""
    def port_open(p):
        with contextlib.closing(socket.socket()) as s:
            s.settimeout(0.5)
            return s.connect_ex(("127.0.0.1", p)) == 0

    if port_open(port):                      # 이미 떠 있으면 재사용
        print(f"♻️  이미 실행 중 (포트 {port})")
        return None

    # 로그는 파일로 — 파이프가 가득 차 서버가 멈추는 일을 막고, 실패 시 원인을 읽을 수 있다
    log_path = os.path.join(tempfile.gettempdir(), f"streamlit_{port}.log")
    log = open(log_path, "w", encoding="utf-8")

    proc = subprocess.Popen(
        [sys.executable, "-m", "streamlit", "run", script,
         "--server.port", str(port),
         "--server.enableCORS", "false",           # iframe은 '다른 주소'로 취급됩니다 —
         "--server.enableXsrfProtection", "false", #   끄지 않으면 화면이 계속 로딩만 됩니다
         "--server.headless", "true"],             # 최초 실행 '이메일 프롬프트'를 건너뜀
        stdout=log, stderr=subprocess.STDOUT,      #   (없으면 입력을 기다리다 조용히 죽음)
    )
    for _ in range(60):                      # 최대 15초, 0.25초 간격 확인
        if proc.poll() is not None:          # 일찍 죽음 → 로그를 보여줌
            log.close()
            print(f"❌ Streamlit이 종료됨 (code {proc.returncode}) — 로그:")
            print(open(log_path, encoding="utf-8").read()[-2000:])
            return proc
        if port_open(port):
            if IN_COLAB:                     # Colab의 localhost는 '내 PC'가 아니라 Colab 서버입니다
                print(f"✅ 프론트엔드 실행됨 (포트 {port}) — 아래 Step 3에서 화면을 띄웁니다")
            else:
                print(f"✅ 프론트엔드: http://localhost:{port}")
            print(f"   (로그: {log_path})")
            return proc
        time.sleep(0.25)
    proc.terminate(); log.close()
    print(f"❌ 15초 내에 포트가 열리지 않음 — 로그:")
    print(open(log_path, encoding="utf-8").read()[-2000:])
    return proc

proc = run_streamlit("frontend/app_pj2.py", port=8501)


✅ 프론트엔드: http://localhost:8501
   (로그: /var/folders/y6/8h835jsn00167fd_c1r04xbr0000gn/T/streamlit_8501.log)


#### 화면 확인 

<img src = "./img/2.png">




---

## 5. 회고

---

회고는 README.md 파일을 참조하세요. 